# 04 — Análise: Qualidade de Dados e Perguntas de NegócioDuas partes: verificação de qualidade por atributo e as consultas que respondem às cincoperguntas formuladas na etapa de objetivo.

In [ ]:
from pyspark.sql import functions as Fimport matplotlib.pyplot as pltd = spark.table("workspace.gold.fluxo_diario_credito_privado")r = spark.table("workspace.gold.resumo_classe_credito_privado")

## Parte 1 — Qualidade de DadosVerificação das cinco dimensões sobre os dados como chegaram na bronze.

In [ ]:
b = spark.table("workspace.bronze.inf_diario_fi")total = b.count()dup = b.groupBy("CNPJ_FUNDO_CLASSE", "ID_SUBCLASSE", "DT_COMPTC").count().filter("count > 1").count()pl_inv = b.filter(F.col("VL_PATRIM_LIQ").cast("double") <= 0).count()cota_inv = b.filter(F.col("VL_QUOTA").isNull() | (F.col("VL_QUOTA").cast("double") <= 0)).count()sem_cot = b.filter(F.col("NR_COTST").cast("int") == 0).count()data_nula = b.filter(F.to_date("DT_COMPTC").isNull()).count()print("UNICIDADE   chaves duplicadas:", dup)print("ACURACIA    patrimonio <= 0:", pl_inv, f"({100*pl_inv/total:.2f}%)")print("COMPLETUDE  cota nula ou <= 0:", cota_inv, f"({100*cota_inv/total:.2f}%)")print("CONSISTENCIA zero cotistas:", sem_cot, f"({100*sem_cot/total:.2f}%)  -> mantidos, classe ativa sem cotista no dia nao e erro")print("CONSISTENCIA data invalida:", data_nula)print()print("bronze:", total, "-> silver:", spark.table("workspace.silver.fato_informe").count())

In [ ]:
# COMPLETUDE da classificação ANBIMA: nulos com causa conhecida# (fundos estruturados e não adaptados, que a ANBIMA não classifica nesse campo)display(spark.table("workspace.bronze.cad_classe")    .groupBy("Classificacao_Anbima").count().orderBy(F.desc("count")).limit(15))

In [ ]:
# INTEGRIDADE REFERENCIAL: classes no cadastro versus classes com informe no períododc = spark.table("workspace.silver.dim_classe").filter("credito_privado")print("marcadas como credito privado no cadastro:", dc.count())print("com informe no periodo:", r.count())print("diferenca:", dc.count() - r.count(),      "-> registradas sem operacao na janela ou encerradas antes de 09/2025")

In [ ]:
# OUTLIERS e descontinuidade: dias com reporte incompletodisplay(d.filter("not reporte_completo")         .select("data", "grandes_ausentes", "n_classes", "pl_total", "captacao_liquida_total")         .orderBy(F.desc("grandes_ausentes")))

### Diagnóstico da descontinuidade de 13/01/2026Comparação do patrimônio por classe entre o dia anterior e o dia da queda. As classes comvalor nulo no segundo dia são as que não reportaram.Observe que o join agrega por classe antes de comparar: a fato tem uma linha por subclasse,e cruzar sem agregar produziria produto cartesiano.

In [ ]:
def pl_no_dia(dia, sufixo):    return (spark.table("workspace.silver.fato_informe")             .filter(f"data = '{dia}'")             .groupBy("cnpj_classe")             .agg(F.sum("patrimonio_liquido").alias("pl_" + sufixo)))antes = pl_no_dia("2026-01-12", "antes")depois = pl_no_dia("2026-01-13", "depois")display(antes.join(depois, "cnpj_classe", "left")    .join(spark.table("workspace.silver.dim_classe").select("cnpj_classe", "nome_classe", "administrador"),          "cnpj_classe", "left")    .withColumn("delta", F.coalesce(F.col("pl_depois"), F.lit(0.0)) - F.col("pl_antes"))    .orderBy("delta").limit(10))

## Parte 2 — Perguntas de Negócio### Pergunta 1 — Evolução do PL e papel da captação líquidaTrês séries: patrimônio bruto, patrimônio excluindo feeders e captação líquida diária.A comparação entre a trajetória do patrimônio e o saldo acumulado de captação distinguecrescimento por entrada de recursos de crescimento por valorização de cota.

In [ ]:
pdf = d.orderBy("data").toPandas()fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(11, 7), sharex=True,                               gridspec_kw={"height_ratios": [2, 1]})ax1.plot(pdf["data"], pdf["pl_total"] / 1e12, lw=1.2, label="PL total (com feeders)")ax1.plot(pdf["data"], pdf["pl_total_ex_feeder"] / 1e12, lw=1.2, label="PL ex-feeder")incompletos = pdf[~pdf["reporte_completo"]]ax1.scatter(incompletos["data"], incompletos["pl_total"] / 1e12,            color="red", s=12, zorder=5, label="reporte incompleto")ax1.set_ylabel("PL agregado (R$ tri)")ax1.set_title("Fundos de crédito privado: patrimônio e captação líquida diária")ax1.legend(fontsize=8)ax1.grid(alpha=0.3)cores = ["tab:red" if v < 0 else "tab:blue" for v in pdf["captacao_liquida_total"]]ax2.bar(pdf["data"], pdf["captacao_liquida_total"] / 1e9, color=cores, width=1.0)ax2.axhline(0, color="grey", lw=0.8)ax2.set_ylabel("Capt. líquida (R$ bi)")ax2.set_xlabel("Data")ax2.grid(alpha=0.3)plt.tight_layout()display(fig)

In [ ]:
# Decomposição: quanto da variação do PL veio de captação líquida acumuladacompletos = d.filter("reporte_completo").orderBy("data").toPandas()pl_ini = completos["pl_total"].iloc[0]pl_fim = completos["pl_total"].iloc[-1]variacao = pl_fim - pl_inicapt_acum = completos["captacao_liquida_total"].sum()print(f"PL inicial:                R$ {pl_ini/1e12:,.3f} tri")print(f"PL final:                  R$ {pl_fim/1e12:,.3f} tri")print(f"Variacao no periodo:       R$ {variacao/1e9:,.1f} bi  ({100*variacao/pl_ini:.1f}%)")print(f"Captacao liquida acumulada:R$ {capt_acum/1e9:,.1f} bi")print(f"Parcela explicada por captacao: {100*capt_acum/variacao:.1f}%")print(f"Parcela explicada por valorizacao (residuo): {100*(variacao-capt_acum)/variacao:.1f}%")

### Pergunta 2 — Porte e estabilidade da base de cotistasCoeficiente de variação do número de cotistas por faixa de patrimônio. Classes com menos de100 dias de histórico são excluídas para não contaminar a medida com séries curtas.

In [ ]:
p2 = (r.filter("cotistas_medio > 0 and dias > 100")   .withColumn("faixa_pl", F.when(F.col("pl_medio") < 1e7, "1. ate 10 mi")                            .when(F.col("pl_medio") < 1e8, "2. 10-100 mi")                            .when(F.col("pl_medio") < 1e9, "3. 100 mi-1 bi")                            .otherwise("4. acima de 1 bi"))   .groupBy("faixa_pl")   .agg(F.avg("cv_cotistas").alias("cv_medio"),        F.expr("percentile_approx(cv_cotistas, 0.5)").alias("cv_mediano"),        F.count("*").alias("n"))   .orderBy("faixa_pl"))display(p2)

In [ ]:
pp = p2.toPandas()fig, ax = plt.subplots(figsize=(8, 4))x = range(len(pp))ax.bar([i - 0.2 for i in x], pp["cv_medio"], width=0.4, label="média")ax.bar([i + 0.2 for i in x], pp["cv_mediano"], width=0.4, label="mediana")ax.set_xticks(list(x))ax.set_xticklabels(pp["faixa_pl"], fontsize=9)ax.set_ylabel("Coef. de variação dos cotistas")ax.set_title("Estabilidade da base de cotistas por porte da classe")ax.legend(fontsize=8)ax.grid(alpha=0.3, axis="y")plt.tight_layout()display(fig)

### Pergunta 3 — Pulverização da base e captação líquidaA média é sensível a valores extremos, então a mediana é apresentada ao lado. O totalcaptado por perfil mostra a magnitude agregada do movimento.

In [ ]:
p3 = (r.filter("dias > 100")   .withColumn("perfil", F.when(F.col("cotistas_medio") < 10, "1. concentrado (< 10)")                          .when(F.col("cotistas_medio") < 1000, "2. intermediario (10-1000)")                          .otherwise("3. pulverizado (> 1000)"))   .groupBy("perfil")   .agg(F.avg("captacao_liquida_periodo").alias("capt_liq_media"),        F.expr("percentile_approx(captacao_liquida_periodo, 0.5)").alias("capt_liq_mediana"),        F.sum("captacao_liquida_periodo").alias("capt_liq_total"),        F.count("*").alias("n"))   .orderBy("perfil"))display(p3)

In [ ]:
pp3 = p3.toPandas()fig, ax = plt.subplots(figsize=(8, 4))cores = ["tab:blue" if v >= 0 else "tab:red" for v in pp3["capt_liq_total"]]ax.bar(pp3["perfil"], pp3["capt_liq_total"] / 1e9, color=cores)ax.axhline(0, color="grey", lw=0.8)ax.set_ylabel("Captação líquida total (R$ bi)")ax.set_title("Captação líquida acumulada por perfil de base de cotistas")ax.grid(alpha=0.3, axis="y")plt.xticks(fontsize=9)plt.tight_layout()display(fig)

### Pergunta 4 — Dispersão de retorno por classificação ANBIMAClasses com menos de 250 dias corridos de histórico são excluídas, para que o retornoacumulado seja comparável entre elas.

In [ ]:
p4 = r.filter("dias_corridos > 250 and retorno_periodo is not null")display(p4.groupBy("classificacao_anbima")    .agg(F.avg("retorno_periodo").alias("retorno_medio"),         F.expr("percentile_approx(retorno_periodo, 0.5)").alias("retorno_mediano"),         F.stddev("retorno_periodo").alias("dispersao"),         F.min("retorno_periodo").alias("minimo"),         F.max("retorno_periodo").alias("maximo"),         F.count("*").alias("n"))    .orderBy(F.desc("n")))

In [ ]:
pdf4 = p4.select("classificacao_anbima", "retorno_periodo").toPandas()fig, ax = plt.subplots(figsize=(10, 4.5))for nome, grupo in pdf4.groupby("classificacao_anbima"):    ax.hist(grupo["retorno_periodo"], bins=60, range=(-0.15, 0.25),            alpha=0.55, label=f"{nome} (n={len(grupo)})")ax.axvline(0, color="grey", lw=0.8)ax.set_xlabel("Retorno acumulado no período (set/25 a jul/26)")ax.set_ylabel("Número de classes")ax.set_title("Dispersão de retorno por classificação ANBIMA")ax.legend(fontsize=8)ax.grid(alpha=0.3)plt.tight_layout()display(fig)

### Pergunta 5 — Dias de resgate atípicoDuas leituras: os maiores dias em valor absoluto e o teste formal do efeito de calendário,usando os atributos da dimensão de tempo. A segunda substitui a inspeção visual por umacomparação de médias.

In [ ]:
display(d.filter("reporte_completo").orderBy(F.desc("resgate_total")).limit(15)   .select("data", "resgate_total", "captacao_liquida_total", "pl_total",           "ultimo_dia_util_mes", "dezembro"))

In [ ]:
# Teste do efeito de calendário: resgate médio por tipo de diadisplay(d.filter("reporte_completo")    .withColumn("tipo_dia",        F.when(F.col("ultimo_dia_util_trimestre"), "3. ultimo dia util do trimestre")         .when(F.col("ultimo_dia_util_mes"), "2. ultimo dia util do mes")         .otherwise("1. dia comum"))    .groupBy("tipo_dia")    .agg(F.avg("resgate_total").alias("resgate_medio"),         F.avg("captacao_liquida_total").alias("capt_liquida_media"),         F.count("*").alias("n_dias"))    .orderBy("tipo_dia"))

In [ ]:
# Dezembro contra o resto do períododisplay(d.filter("reporte_completo")    .groupBy("dezembro")    .agg(F.avg("resgate_total").alias("resgate_medio"),         F.avg("captacao_liquida_total").alias("capt_liquida_media"),         F.count("*").alias("n_dias")))

---A discussão dos resultados, as ressalvas metodológicas e a autoavaliação estão no`README.md` do repositório.